# Mini Project 1 — Analysis Notebook

**Your name:** Aleigha Mattison 
**Dataset:**  Zillow Housing Data
**Date:**  5.20.26

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px


from pathlib import Path
import os

def _find_data_dir():
    """Use the folder that contains this notebook's CSV files."""
    cwd = Path.cwd()
    for candidate in (cwd, cwd / 'Mini project 1'):
        if (candidate / 'listings.csv').exists():
            return candidate
    return cwd

os.chdir(_find_data_dir())
print(f'Working directory: {Path.cwd().resolve()}')

def data_quality_check(csv_path=None):
    """Print shape, missing values, dtypes, outliers, and string issues for df."""
    global df
    if csv_path is not None:
        df = pd.read_csv(csv_path)
        print(f'Loaded: {csv_path}  shape: {df.shape}')
    elif 'df' not in globals():
        raise RuntimeError(
            'df is not defined. Run the Setup cell first, then either the '
            'pd.read_csv(...) cell above or call data_quality_check(csv_path=...).'
        )

    print('Data quality check')
    print('shape:', df.shape)
    print('duplicate cols:', any(df.columns.duplicated()))
    missing = df.isna().sum()
    print('total missing values:', missing.sum())
    print('columns with missing values:', (missing > 0).sum())
    print('dtypes:')
    print(df.dtypes.value_counts().to_dict())
    nums = df.select_dtypes(include=['number'])
    if not nums.empty:
        z = (nums - nums.mean()) / nums.std(ddof=0)
        print('numeric outliers >3σ:', int((z.abs() > 3).sum().sum()))
    else:
        print('numeric outliers >3σ: none')
    strings = df.select_dtypes(include=['object', 'string'])
    if not strings.empty:
        bad = []
        for col in strings.columns:
            if strings[col].dropna().astype(str).str.contains(r'^\s+|\s+$|\s{2,}|\r|\n', na=False).any():
                bad.append(col)
        print('string formatting issues:', bad if bad else 'none')
    else:
        print('string formatting issues: none')

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** *(What is it? Where did it come from? Paste the URL or citation from your MP1a submission.)* 

This is the Zillow Housing data set which is compiled by Zillow. There are different datasets available coverinng things like home listing, rental data, indeces and forecasts.
URL: https://www.zillow.com/research/data/

**Why this dataset:** *(One sentence connecting it to your HCD work or research interests.)*
This is a valuable dataset and analysis for the HCD field as housing is an inherently human-centered problem space, as everyone either rents or buys a place to live, making insights into it essential.

**Three analytical questions:**

1. *(Question 1 from MP1a)* Are there seasonal patterns in listins counts, sales counts and price for houses in the Seattle metro area that could inform my purchase timing? 
2. *(Question 2 from MP1a)* Over the last ten years, by what percentage has ZORI increased in my neighborhood, and how does that compare to the metro area and national ZORI growth? What does this show about the market?
3. *(Question 3 from MP1a)* How has ZORDI changed in my neighborhood over the last ten years, and how does that pattern compare to ZORI over the same period? What does this show about the market?

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

## Section 1.1 — Overview

- I started with reviewing my datasets more deeply to understand what visualizations I could build. 
I then learned that I could not see a breakdown of 2-3 bedroom houses for all data like I initially incorrectly assumed. I could for home values while still seeing the zip code though. Additionally, condos were not always broken out so I had to settle for joint categories sometimes and adjust my research questions from there. Continuting reviewing the datasets, sometimes there was not a zip code breakdown for different datasets which I found most useful as I was looking for information closest to my neighborhood. This happened wiwth listings and sales which onyl had a breakdown by metro areas. 

When I looked into housing values, I discovered to answer questions on seasonality I needed the RAW versions. After investigating I found out this is the unfiltered, monthly changes in estimated home values whereas the other option is is a moving average and adjsuts out seasonal values. As I was specifically looking for seasonaility this would not be useful. 

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [6]:
df = pd.read_csv('Zip_zhvi_bdrmcnt_2_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (1).csv')  # ← replace with your filename

print(df.shape)
df.head()


(16000, 325)


,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2000-01-31,...,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,NaN,...,360508.218151,359357.323120,358704.192945,357914.487019,357184.945458,356313.526325,355192.552396,353542.180707,351726.204961,349977.121803
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,75652.151659,...,258365.902246,257410.661725,256544.844566,255900.940533,254875.738564,253658.789290,252401.819238,251400.152344,250737.754086,250114.428263
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,63819.647072,...,192097.810874,190853.219492,189734.280749,188790.664897,188397.012546,188305.277242,188120.935653,188011.459343,188163.137927,188171.748680
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,141898.776205,...,425081.251358,426154.298979,428197.521766,430548.467054,432944.893336,435824.040355,439653.718978,443615.620797,447754.078489,450070.676268
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,64589.223296,...,180073.533662,179359.250123,178410.259505,177504.736300,176830.130319,176548.681661,176132.955270,175823.070474,175614.705412,175108.513906


### Data quality check for the zip-level 2-bedroom ZHVI dataset
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Zip_zhvi_bdrmcnt_2_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (1).csv')


- How many rows and columns does your dataset have? 
It has 324 rows and 16,000 columns 

- What does each column represent?
Region ID is a unique identifier for the region. Size Rank indicates the population ranking of the area. Region Name is the zip code. Region Type specifies that it is a zip code. State Name is the state abbreviation. City is the city. Metro indicates the metro area. County Name is the county. After that, each column is a monthly breakdown of values starting in the year 2000.

- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [8]:
df = pd.read_csv('listings.csv')  # ← replace with your filename

print(df.shape)
df.head()

(928, 102)


,RegionID,SizeRank,RegionName,RegionType,StateName,2018-03-31,2018-04-30,2018-05-31,2018-06-30,2018-07-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,1421529.0,1500195.0,1592417.0,1660618.0,1709146.0,...,1329558.0,1373264.0,1380205.0,1373116.0,1362806.0,1322706.0,1239517.0,1158763.0,1115629.0,1155079.0
1,394913,1,"New York, NY",msa,NY,73707.0,80345.0,85864.0,90067.0,91881.0,...,47708.0,49038.0,48097.0,47265.0,46782.0,45632.0,41945.0,38103.0,35740.0,37158.0
2,753899,2,"Los Angeles, CA",msa,CA,21998.0,23784.0,25605.0,27109.0,28811.0,...,25128.0,26379.0,26688.0,26539.0,25801.0,24240.0,21853.0,20066.0,19707.0,21131.0
3,394463,3,"Chicago, IL",msa,IL,38581.0,42253.0,45757.0,47492.0,48984.0,...,23133.0,24026.0,24129.0,24076.0,23989.0,22985.0,20628.0,18354.0,17282.0,18258.0
4,394514,4,"Dallas, TX",msa,TX,24042.0,25876.0,28224.0,30490.0,32408.0,...,37501.0,38954.0,39091.0,38265.0,37323.0,35777.0,33494.0,31310.0,30185.0,31238.0


### Data quality check for listings.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='listings.csv')


In [10]:
df = pd.read_csv('sales.csv')  # ← replace with your filename

print(df.shape)
df.head()

(301, 223)


,RegionID,SizeRank,RegionName,RegionType,StateName,2008-02-29,2008-03-31,2008-04-30,2008-05-31,2008-06-30,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,203822.0,235967.0,261747.0,288767.0,301790.0,...,359536.0,357359.0,341460.0,326838.0,332061.0,263951.0,303764.0,217693.0,235352.0,305061.0
1,394913,1,"New York, NY",msa,NY,8526.0,9055.0,10043.0,10470.0,11319.0,...,13598.0,14854.0,14364.0,13799.0,13639.0,11015.0,13184.0,10394.0,8466.0,10271.0
2,753899,2,"Los Angeles, CA",msa,CA,4135.0,5052.0,6078.0,6865.0,7220.0,...,6622.0,6985.0,6499.0,6645.0,7009.0,5395.0,6135.0,4432.0,5017.0,6589.0
3,394463,3,"Chicago, IL",msa,IL,5601.0,6961.0,7327.0,7992.0,8824.0,...,10412.0,10147.0,9277.0,8541.0,8770.0,6581.0,7432.0,5188.0,5513.0,8159.0
4,394514,4,"Dallas, TX",msa,TX,4918.0,5588.0,6030.0,6727.0,6720.0,...,7071.0,7148.0,6585.0,6063.0,6044.0,4749.0,5751.0,4113.0,4967.0,6550.0


### Data quality check for sales.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='sales.csv')


In [2]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual filename.
# The file should be in the same folder as this notebook.
# If you're loading from an API result, replace pd.read_csv() with the appropriate call.
#
# Example (app review dataset from class):
# df = pd.read_csv('app_reviews_demo.csv')

df = pd.read_csv('Metro_median_sale_price_now_uc_sfrcondo_month.csv')  # ← replace with your filename

print(df.shape)
df.head()

(301, 223)


,RegionID,SizeRank,RegionName,RegionType,StateName,2008-02-29,2008-03-31,2008-04-30,2008-05-31,2008-06-30,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,170250.0,175000.0,177000.0,180000.0,185000.0,...,375000.0,370000.0,368000.0,360000.0,362500.0,360000.0,355000.0,350000.0,357000.0,369494.0
1,394913,1,"New York, NY",msa,NY,400000.0,390000.0,390000.0,392000.0,400000.0,...,685000.0,686000.0,685000.0,660000.0,654250.0,665000.0,650000.0,655000.0,650000.0,646468.0
2,753899,2,"Los Angeles, CA",msa,CA,470000.0,455000.0,457750.0,440000.0,435000.0,...,1000000.0,960000.0,949000.0,955000.0,945000.0,945000.0,923000.0,920000.0,950000.0,974029.0
3,394463,3,"Chicago, IL",msa,IL,218750.0,220000.0,223000.0,229000.0,235000.0,...,350000.0,340000.0,340000.0,325000.0,325000.0,324000.0,315000.0,314000.0,320000.0,331995.0
4,394514,4,"Dallas, TX",msa,TX,138000.0,145500.0,145000.0,150000.0,156000.0,...,408000.0,400000.0,387000.0,380000.0,380000.0,374100.0,370000.0,370000.0,379500.0,390123.0


### Data quality check for Metro_median_sale_price_now_uc_sfrcondo_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Metro_median_sale_price_now_uc_sfrcondo_month.csv')


In [14]:
df = pd.read_csv('Zori_zip_uc_sfrcondomfr_sm_month.csv')  # ← A smoothed measure of the typical observed market rate rent across a given region. ZORI is a repeat-rent index that is weighted to the rental housing stock to ensure representativeness across the entire market, not just those homes currently listed for-rent. 
print(df.shape)
df.head()

(8188, 144)


,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2015-01-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,1396.420611,...,1752.349869,1761.399334,1768.181320,1762.748802,1752.914282,1739.839933,1735.345787,1716.202697,1704.095737,1683.654847
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2305.088360,2379.926927,2310.000000
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,1238.667257,...,1825.135558,1823.161847,1814.078021,1805.419803,1804.616865,1793.828474,1804.151902,1793.823253,1797.458821,1779.454743
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2700.000000
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,1093.863007,...,1576.421608,1573.457268,1579.511547,1579.319611,1574.482724,1564.271077,1559.644039,1559.997423,1554.930010,1544.184291


### Data quality check for Zori_zip_uc_sfrcondomfr_sm_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Zori_zip_uc_sfrcondomfr_sm_month.csv')


In [15]:
df = pd.read_csv('Zori_counties_uc_sfrcondomfr_sm_month.csv')  # ← A smoothed measure of the typical observed market rate rent across a given region. ZORI is a repeat-rent index that is weighted to the rental housing stock to ensure representativeness across the entire market, not just those homes currently listed for-rent. 
print(df.shape)
df.head()

(1314, 144)


,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2015-01-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,1689.825825,...,2785.991155,2790.526918,2791.479802,2792.557959,2789.774875,2782.151346,2771.997035,2773.905208,2780.781949,2793.537619
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,1445.605077,...,2196.888668,2208.326886,2212.349504,2209.270715,2201.604643,2195.470066,2192.535578,2207.166490,2228.838220,2256.265647
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,1170.994117,...,1595.935882,1596.978077,1596.286491,1598.689508,1592.015635,1584.536583,1578.330112,1576.480908,1574.662778,1576.783883
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,914.557502,...,1745.899459,1744.246654,1739.928274,1736.304903,1729.357332,1720.561859,1715.215394,1715.558139,1722.897256,1728.445033
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,1594.050651,...,2880.106700,2885.472824,2891.367609,2888.561217,2884.632568,2869.433489,2861.794719,2864.481514,2877.890526,2889.899307


### Data quality check for zori_counties_uc_sfrcondomfr_sm_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Zori_counties_uc_sfrcondomfr_sm_month.csv')


In [16]:
df = pd.read_csv('Zori_Metro_uc_sfrcondomfr_sm_month.csv')  # ← A smoothed measure of the typical observed market rate rent across a given region. ZORI is a repeat-rent index that is weighted to the rental housing stock to ensure representativeness across the entire market, not just those homes currently listed for-rent. 
print(df.shape)
df.head()

(719, 140)


,RegionID,SizeRank,RegionName,RegionType,StateName,2015-01-31,2015-02-28,2015-03-31,2015-04-30,2015-05-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,1136.747272,1143.002520,1151.566003,1160.295331,1168.904681,...,1901.444662,1905.091763,1906.236631,1904.921138,1901.310487,1895.636662,1891.140259,1892.439923,1899.581978,1910.424994
1,394913,1,"New York, NY",msa,NY,2184.191044,2198.930168,2217.824385,2236.403118,2250.771613,...,3290.539695,3320.703116,3341.252886,3338.784221,3326.153715,3305.460325,3293.370385,3290.776879,3308.005112,3337.139888
2,753899,2,"Los Angeles, CA",msa,CA,1739.529369,1750.819968,1765.883551,1780.754656,1795.593017,...,2884.341555,2888.500950,2890.075156,2889.810116,2886.795063,2879.898972,2870.937038,2873.470854,2880.684896,2894.901157
3,394463,3,"Chicago, IL",msa,IL,1374.256131,1381.587911,1391.546391,1401.007023,1410.826622,...,2126.664004,2137.955170,2142.052174,2140.203901,2134.356115,2129.121344,2127.097179,2138.409168,2157.030613,2179.817593
4,394514,4,"Dallas, TX",msa,TX,1049.220308,1053.961615,1061.339334,1072.453106,1081.657882,...,1665.128879,1663.359463,1659.929336,1654.747658,1648.248483,1641.725612,1635.817921,1633.592707,1637.284321,1644.537859


### Data quality check for Zori_Metro_uc_sfrcondomfr_sm_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Zori_Metro_uc_sfrcondomfr_sm_month.csv')


In [26]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


In [27]:
# Summary statistics for numeric columns
df.describe()

,id,rating,helpful_votes
count,500.000000,500.000000,500.000000
mean,250.500000,3.946000,23.464000
std,144.481833,1.184013,13.766471
min,1.000000,1.000000,0.000000
25%,125.750000,3.000000,11.000000
50%,250.500000,4.000000,23.500000
75%,375.250000,5.000000,35.000000
max,500.000000,5.000000,47.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *(paste your first research question from MP1a here)*

In [ ]:
# Over the last three years, by what percentage has ZORI increased in my neighborhood (98004), and how does that compare to the metro area and national ZORI growth?


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2:** *(paste your second research question here)*

In [29]:
# Your analysis for Question 2


**Interpretation:**  
*(What does this result tell you?)*

**Question 3:** *(paste your third research question here)*

In [30]:
# Your analysis for Question 3


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [23]:
fig.update_layout(
    title_text="ZORI Growth Analysis: 98004 Zip Code vs Seattle Metro vs National Average (2016-2026)",
    height=500,
    width=1400,
    showlegend=True,
    hovermode='x unified',
    barmode='group',
    legend=dict(
        x=1.05,  # Position legend outside the chart area to the right
        y=0.5,
        xanchor='left',
        yanchor='middle'
    )
)

In [24]:
# ZORI vs ZORDI Comparison for Seattle Metro Area (2016-2026)
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load ZORI and ZORDI data
zori_metro = pd.read_csv('Zori_Metro_uc_sfrcondomfr_sm_month.csv')
zordi_metro = pd.read_csv('Zordi_Metro_uc_condo_month.csv')

# Filter for Seattle metro area
seattle_zori = zori_metro[zori_metro['RegionName'] == 'Seattle, WA'].copy()
seattle_zordi = zordi_metro[zordi_metro['RegionName'] == 'Seattle, WA'].copy()

# Get date columns (starting from 2016)
zori_date_cols = [col for col in seattle_zori.columns if col.startswith('20') and col >= '2016-01-31']
zordi_date_cols = [col for col in seattle_zordi.columns if col.startswith('20') and col >= '2016-01-31']

# Melt ZORI data
zori_long = seattle_zori.melt(
    id_vars=['RegionID', 'RegionName', 'RegionType', 'StateName'],
    value_vars=zori_date_cols,
    var_name='Date',
    value_name='ZORI'
)

# Melt ZORDI data
zordi_long = seattle_zordi.melt(
    id_vars=['RegionID', 'RegionName', 'RegionType', 'StateName'],
    value_vars=zordi_date_cols,
    var_name='Date',
    value_name='ZORDI'
)

# Convert dates and merge
zori_long['Date'] = pd.to_datetime(zori_long['Date'])
zordi_long['Date'] = pd.to_datetime(zordi_long['Date'])

# Merge the datasets
merged = pd.merge(zori_long[['Date', 'ZORI']], zordi_long[['Date', 'ZORDI']], on='Date', how='inner')
merged = merged.sort_values('Date')

# Create dual-axis subplot
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add ZORI trace (left axis - rent prices)
fig.add_trace(
    go.Scatter(
        x=merged['Date'], 
        y=merged['ZORI'], 
        name="ZORI (Rent Index)",
        line=dict(color='blue', width=2)
    ),
    secondary_y=False,
)

# Add ZORDI trace (right axis - demand index)
fig.add_trace(
    go.Scatter(
        x=merged['Date'], 
        y=merged['ZORDI'], 
        name="ZORDI (Demand Index)",
        line=dict(color='red', width=2)
    ),
    secondary_y=True,
)

# Update layout
fig.update_layout(
    title_text="ZORI vs ZORDI Trends in Seattle Metro Area (2016-2026)",
    height=500,
    width=1000,
    hovermode='x unified'
)

# Update axes labels
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="ZORI (Rent Index - $)", secondary_y=False)
fig.update_yaxes(title_text="ZORDI (Demand Index)", secondary_y=True)

# Show the chart
fig.show()

### Data quality check for Zori_Metro_uc_sfrcondomfr_sm_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Zori_Metro_uc_sfrcondomfr_sm_month.csv')


In [11]:
# Seattle vs national listings and sales comparison
import pandas as pd
import plotly.express as px

listings = pd.read_csv('listings.csv')
sales = pd.read_csv('sales.csv')

listings_date_cols = [c for c in listings.columns if c.startswith('20') and '-' in c]
sales_date_cols = [c for c in sales.columns if c.startswith('20') and '-' in c]

listings_long = listings.melt(
    id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'],
    value_vars=listings_date_cols,
    var_name='Date',
    value_name='Count'
)
sales_long = sales.melt(
    id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'],
    value_vars=sales_date_cols,
    var_name='Date',
    value_name='Count'
)

listings_long['Date'] = pd.to_datetime(listings_long['Date'])
sales_long['Date'] = pd.to_datetime(sales_long['Date'])

start_date = pd.Timestamp('2016-01-01')
listings_long = listings_long[listings_long['Date'] >= start_date]
sales_long = sales_long[sales_long['Date'] >= start_date]

seattle_listings = listings_long[
    (listings_long['RegionName'] == 'Seattle, WA') &
    (listings_long['RegionType'] == 'msa')
].copy()
seattle_sales = sales_long[
    (sales_long['RegionName'] == 'Seattle, WA') &
    (sales_long['RegionType'] == 'msa')
].copy()

national_listings = listings_long[listings_long['RegionType'] != 'country']
national_sales = sales_long[sales_long['RegionType'] != 'country']

national_listings = national_listings.groupby('Date')['Count'].mean().reset_index()
national_sales = national_sales.groupby('Date')['Count'].mean().reset_index()

seattle_listings['Series'] = 'Seattle Listings'
seattle_sales['Series'] = 'Seattle Sales'
national_listings['Series'] = 'National Avg Listings'
national_sales['Series'] = 'National Avg Sales'

plot_df = pd.concat([
    seattle_listings[['Date', 'Count', 'Series']],
    seattle_sales[['Date', 'Count', 'Series']],
    national_listings[['Date', 'Count', 'Series']],
    national_sales[['Date', 'Count', 'Series']]
], ignore_index=True)

fig = px.line(
    plot_df.sort_values(['Series', 'Date']),
    x='Date',
    y='Count',
    color='Series',
    title='Seattle Listings and Sales vs National Average (Last 10 Years)',
    labels={
        'Date': 'Date',
        'Count': 'Number of Homes',
        'Series': 'Series'
    }
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Number of Homes',
    legend_title='Series',
    template='plotly_white'
)

fig.show()


### Data quality check for listings.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='listings.csv')


**Seattle vs national listings and sales comparison:**  
This chart compares the number of homes listed for sale and sold in Seattle, WA over the last 10 years with the national average across all regions.  

**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

### Seasonality Analysis: Seattle vs. West Coast Comparable Cities

This visualization analyzes whether there are seasonal patterns in home pricing for Seattle and benchmarks it against three comparable West Coast cities: San Francisco, Portland, and San Diego. We calculate the average median price for each month across the 10-year period (2016-2026) to identify recurring seasonality patterns. This helps answer: Do homes have predictable price fluctuations based on the month of the year?

In [ ]:
# Load metro price data
metro_data = pd.read_csv('Metro_median_sale_price_now_uc_sfrcondo_month.csv')

# Filter for the four cities and the last 10 years (2016-2026)
cities = ['Seattle, WA', 'San Francisco, CA', 'Portland, OR', 'San Diego, CA']
city_data = metro_data[metro_data['RegionName'].isin(cities)]

# Melt the dataframe to get it in long format
# Only select date columns from 2016 onwards
date_cols = [col for col in metro_data.columns if col.startswith('201') or col.startswith('202')]
date_cols_filtered = [col for col in date_cols if col >= '2016-01-31']

melted = city_data[['RegionName'] + date_cols_filtered].melt(id_vars='RegionName', var_name='Date', value_name='Price')
melted['Date'] = pd.to_datetime(melted['Date'])
melted['Month'] = melted['Date'].dt.month
melted['Month_Name'] = melted['Date'].dt.strftime('%B')

# Calculate average price by month for each city
seasonality = melted.groupby(['RegionName', 'Month', 'Month_Name'])['Price'].mean().reset_index()
seasonality = seasonality.sort_values('Month')

# Create the visualization
fig = px.line(
    seasonality,
    x='Month_Name',
    y='Price',
    color='RegionName',
    title='Seasonality in Home Pricing: Seattle vs. Comparable West Coast Cities (2016-2026)',
    labels={'Price': 'Average Median Sale Price ($)', 'Month_Name': 'Month', 'RegionName': 'City'},
    markers=True,
    line_shape='linear'
)

fig.update_layout(
    hovermode='x unified',
    height=500,
    xaxis_tickangle=-45,
    template='plotly_white',
    font=dict(size=12),
    title_font_size=14
)

fig.show()

# Print summary statistics
print('\nSeasonality Summary by City:')
print('=' * 70)
for city in cities:
    city_seasonal = seasonality[seasonality['RegionName'] == city]
    max_price = city_seasonal['Price'].max()
    min_price = city_seasonal['Price'].min()
    max_month = city_seasonal[city_seasonal['Price'] == max_price]['Month_Name'].values[0]
    min_month = city_seasonal[city_seasonal['Price'] == min_price]['Month_Name'].values[0]
    variance = max_price - min_price
    pct_variance = (variance / min_price) * 100
    
    print(f'\n{city}:')
    print(f'  Highest avg price: ${max_price:,.0f} ({max_month})')
    print(f'  Lowest avg price:  ${min_price:,.0f} ({min_month})')
    print(f'  Price variance:     ${variance:,.0f} ({pct_variance:.1f}%)')

### Data quality check for Metro_median_sale_price_now_uc_sfrcondo_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Metro_median_sale_price_now_uc_sfrcondo_month.csv')


In [5]:
# Your visualization
import pandas as pd
import plotly.express as px

# Step 1: Load the 2-bedroom zip-level ZHVI data
df_2bed = pd.read_csv('Zip_zhvi_bdrmcnt_2_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (1).csv')

# Identify the date columns (from 2000-01-31 to 2026-03-31)
date_cols = [col for col in df_2bed.columns if col.startswith('20') and '-' in col]

# Melt the dataframe to long format
df_long = df_2bed.melt(
    id_vars=['RegionID', 'RegionName', 'RegionType', 'City', 'State'],
    value_vars=date_cols,
    var_name='Date',
    value_name='ZHVI'
)

# Convert Date to datetime
df_long['Date'] = pd.to_datetime(df_long['Date'])

# Filter for the last 10 years (2016 onwards)
df_filtered = df_long[df_long['Date'] >= '2016-01-01']

# Step 2: Filter for comparison regions
bellevue_overall = df_filtered[df_filtered['City'] == 'Bellevue'].copy()
bellevue_overall['Region_Group'] = 'Bellevue (Overall)'

bellevue_98004 = df_filtered[(df_filtered['RegionName'] == '98004') & (df_filtered['RegionType'] == 'zip')].copy()
bellevue_98004['Region_Group'] = 'Bellevue (98004)'

seattle_city = df_filtered[(df_filtered['City'] == 'Seattle') & (df_filtered['RegionType'] == 'city')].copy()
seattle_city['Region_Group'] = 'Seattle (City)'

portland_city = df_filtered[(df_filtered['City'] == 'Portland') & (df_filtered['RegionType'] == 'city')].copy()
portland_city['Region_Group'] = 'Portland (City)'

washington_state = df_filtered[(df_filtered['State'] == 'WA') & (df_filtered['RegionType'] == 'zip')].copy()
washington_state['Region_Group'] = 'Washington (Zip Average)'

oregon_state = df_filtered[(df_filtered['State'] == 'OR') & (df_filtered['RegionType'] == 'zip')].copy()
oregon_state['Region_Group'] = 'Oregon (Zip Average)'

# Combine the filtered data
df_plot = pd.concat([
    bellevue_overall,
    bellevue_98004,
    seattle_city,
    portland_city,
    washington_state,
    oregon_state
], ignore_index=True)

# Step 3: Aggregate by date and region group (average ZHVI per month)
df_agg = df_plot.groupby(['Date', 'Region_Group'])['ZHVI'].mean().reset_index()

# Step 4: Create the line chart
fig = px.line(
    df_agg,
    x='Date',
    y='ZHVI',
    color='Region_Group',
    title='2-Bedroom Home Value Trends Over the Last 10 Years: Bellevue, Seattle, Portland, and WA/OR Averages',
    labels={
        'Date': 'Date',
        'ZHVI': 'ZHVI (2-Bedroom Median Home Value Estimate)',
        'Region_Group': 'Region'
    }
)

# Customize the layout
fig.update_layout(
    xaxis_title='Year',
    yaxis_title='ZHVI (2-Bedroom Median Home Value Estimate)',
    legend_title='Region',
    template='plotly_white'
)

# Show the chart
fig.show()


### Data quality check for the zip-level 2-bedroom ZHVI dataset
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Zip_zhvi_bdrmcnt_2_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (1).csv')


---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

**C6 — Data Visualization**  
I built line charts in Python to make a specific argument about market trends, comparing ZORI and ZORDI over time and adding a listings vs sales comparison to show whether listed homes were actually selling. I chose line charts because the data are monthly time series and the visual structure clearly shows trend direction, timing, and relative growth. I also expanded the analysis window to 10 years so the three-year neighborhood and metro comparisons are placed in a broader context, rather than relying only on short-term variation. This notebook is published on GitHub with code cells, chart output, and markdown explanations that document the findings and why the chart choices were appropriate.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.